# IMPORT LIBRARIES

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
pd.reset_option("all")
import os
import pyarrow.parquet as pq

C:\Users\AdminOS\AppData\Local\Temp\ipykernel_23136\1633080496.py:6: FutureWarning: data_manager option is deprecated and will be removed in a future version. Only the BlockManager will be available.
  pd.reset_option("all")
C:\Users\AdminOS\AppData\Local\Temp\ipykernel_23136\1633080496.py:6: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  pd.reset_option("all")


# CONNECT TO THE DATABASE

In [3]:
# Database Connection
DB_USER = "stock_user"
DB_PASSWORD = "stock_pass"
DB_HOST = "localhost"
DB_PORT = "5433"
DB_NAME = "stock_db"

connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# create SQLAlchemy engine
engine = create_engine(connection_string)

# test connection
try:
    with engine.connect() as conn:
        print("✅ Connected to database")
except Exception as e:
    print("❌ Connection failed:", e)

✅ Connected to database


# Data VAlidation

In [4]:
# LIST OF TABLES TO LOAD
tables = ["market_1h", "market_1d", "market_15m"]

# DATA VALIDATION & LOADING
for table in tables:
    print(f"\nLoading table: {table}")
    query = f"SELECT * FROM {table}"
    df = pd.read_sql(query, engine)
    
    print("Shape:", df.shape)
    display(df.head())


Loading table: market_1h
Shape: (44893, 9)


,ts,open,high,low,close,adj_close,volume,symbol,interval
0,2024-02-26 05:00:00,NaN,NaN,NaN,NaN,None,NaN,GC=F,1h
1,2024-02-26 06:00:00,NaN,NaN,NaN,NaN,None,NaN,GC=F,1h
2,2024-02-26 07:00:00,NaN,NaN,NaN,NaN,None,NaN,GC=F,1h
3,2024-02-26 08:00:00,NaN,NaN,NaN,NaN,None,NaN,GC=F,1h
4,2024-02-26 09:00:00,NaN,NaN,NaN,NaN,None,NaN,GC=F,1h



Loading table: market_1d
Shape: (44244, 9)


,ts,open,high,low,close,adj_close,volume,symbol,interval
0,2010-01-04,21.680000,21.680000,20.030001,20.040001,None,0.0,^VIX,1d
1,2010-01-05,20.049999,20.129999,19.340000,19.350000,None,0.0,^VIX,1d
2,2010-01-06,19.590000,19.680000,18.770000,19.160000,None,0.0,^VIX,1d
3,2010-01-07,19.680000,19.709999,18.700001,19.059999,None,0.0,^VIX,1d
4,2010-01-08,19.270000,19.270000,18.110001,18.129999,None,0.0,^VIX,1d



Loading table: market_15m
Shape: (18341, 9)


,ts,open,high,low,close,adj_close,volume,symbol,interval
0,2025-12-28 23:00:00,NaN,NaN,NaN,NaN,None,NaN,CL=F,15m
1,2025-12-28 23:15:00,NaN,NaN,NaN,NaN,None,NaN,CL=F,15m
2,2025-12-28 23:30:00,NaN,NaN,NaN,NaN,None,NaN,CL=F,15m
3,2025-12-28 23:45:00,NaN,NaN,NaN,NaN,None,NaN,CL=F,15m
4,2025-12-29 00:00:00,NaN,NaN,NaN,NaN,None,NaN,CL=F,15m


there are many missing values, so I want to display only the non-null values

In [5]:
# Data validation: load tables and skip rows with any null values in key columns
for table in tables:
    print(f"\nLoading table: {table}")
    
    query = f"""
    SELECT *
    FROM {table}
    WHERE ts IS NOT NULL
      AND open IS NOT NULL
      AND high IS NOT NULL
      AND low IS NOT NULL
      AND close IS NOT NULL
      AND volume IS NOT NULL
      AND symbol IS NOT NULL
    """
    
    df = pd.read_sql(query, engine)
    print("Shape after skipping nulls:", df.shape)
    display(df.head())



Loading table: market_1h
Shape after skipping nulls: (21639, 9)


,ts,open,high,low,close,adj_close,volume,symbol,interval
0,2024-02-26 14:30:00,39144.789062,39245.890625,39135.230469,39195.929688,None,0,^DJI,1h
1,2024-02-26 15:30:00,39195.589844,39213.390625,39119.121094,39142.339844,None,35336642,^DJI,1h
2,2024-02-26 16:30:00,39141.121094,39162.109375,39121.210938,39156.910156,None,29436849,^DJI,1h
3,2024-02-26 17:30:00,39155.968750,39161.378906,39070.660156,39084.621094,None,23371225,^DJI,1h
4,2024-02-26 18:30:00,39084.738281,39129.851562,39025.800781,39119.460938,None,23883044,^DJI,1h



Loading table: market_1d
Shape after skipping nulls: (20388, 9)


,ts,open,high,low,close,adj_close,volume,symbol,interval
0,2010-01-04,21.680000,21.680000,20.030001,20.040001,None,0,^VIX,1d
1,2010-01-05,20.049999,20.129999,19.340000,19.350000,None,0,^VIX,1d
2,2010-01-06,19.590000,19.680000,18.770000,19.160000,None,0,^VIX,1d
3,2010-01-07,19.680000,19.709999,18.700001,19.059999,None,0,^VIX,1d
4,2010-01-08,19.270000,19.270000,18.110001,18.129999,None,0,^VIX,1d



Loading table: market_15m
Shape after skipping nulls: (8546, 9)


,ts,open,high,low,close,adj_close,volume,symbol,interval
0,2025-12-29 14:30:00,48617.281250,48704.828125,48569.699219,48570.378906,None,15977968,^DJI,15m
1,2025-12-29 14:45:00,48570.781250,48609.691406,48531.980469,48568.179688,None,17147909,^DJI,15m
2,2025-12-29 15:00:00,48566.488281,48579.070312,48510.058594,48521.738281,None,12950107,^DJI,15m
3,2025-12-29 15:15:00,48521.011719,48548.121094,48506.699219,48515.980469,None,11077896,^DJI,15m
4,2025-12-29 15:30:00,48515.441406,48525.101562,48484.898438,48519.210938,None,11299688,^DJI,15m


From these tables, I noticed that the adj_close column is always null.

In [6]:
for table in tables:
    query = f"SELECT * FROM {table}"
    df = pd.read_sql(query, engine)
    print(f"\nInfo for table: {table}")
    display(df.info())


Info for table: market_1h
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44893 entries, 0 to 44892
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   ts         44893 non-null  datetime64[ns]
 1   open       21639 non-null  float64       
 2   high       21639 non-null  float64       
 3   low        21639 non-null  float64       
 4   close      21639 non-null  float64       
 5   adj_close  0 non-null      object        
 6   volume     21639 non-null  float64       
 7   symbol     44893 non-null  object        
 8   interval   44893 non-null  object        
dtypes: datetime64[ns](1), float64(5), object(3)
memory usage: 3.1+ MB


None


Info for table: market_1d
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44244 entries, 0 to 44243
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   ts         44244 non-null  datetime64[ns]
 1   open       20388 non-null  float64       
 2   high       20388 non-null  float64       
 3   low        20388 non-null  float64       
 4   close      20388 non-null  float64       
 5   adj_close  0 non-null      object        
 6   volume     20388 non-null  float64       
 7   symbol     44244 non-null  object        
 8   interval   44244 non-null  object        
dtypes: datetime64[ns](1), float64(5), object(3)
memory usage: 3.0+ MB


None


Info for table: market_15m
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18341 entries, 0 to 18340
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   ts         18341 non-null  datetime64[ns]
 1   open       8546 non-null   float64       
 2   high       8546 non-null   float64       
 3   low        8546 non-null   float64       
 4   close      8546 non-null   float64       
 5   adj_close  0 non-null      object        
 6   volume     8546 non-null   float64       
 7   symbol     18341 non-null  object        
 8   interval   18341 non-null  object        
dtypes: datetime64[ns](1), float64(5), object(3)
memory usage: 1.3+ MB


None

In [7]:
for table in tables:

    query = f"SELECT * FROM {table}"
    df = pd.read_sql(query, engine)
    print(f"\nDescription for table: {table}")
    display(df.describe())


Description for table: market_1h


,ts,open,high,low,close,volume
count,44893,21639.000000,21639.000000,21639.000000,21639.000000,2.163900e+04
mean,2025-03-07 18:23:20.369767680,11373.210843,11395.430451,11350.635250,11373.429907,1.474789e+08
min,2024-02-26 05:00:00,3.614000,3.631000,3.603000,3.621000,0.000000e+00
25%,2024-08-30 18:00:00,15.060000,15.190000,14.930000,15.040000,0.000000e+00
50%,2025-03-10 14:00:00,32.590000,33.349998,31.809999,32.470001,0.000000e+00
75%,2025-09-10 16:00:00,19156.953125,19192.741211,19115.561523,19158.211914,2.269040e+08
max,2026-03-17 23:00:00,50411.960938,50512.789062,50318.578125,50412.660156,6.594401e+09
std,NaN,15884.209064,15913.637671,15854.894643,15884.678579,2.775594e+08



Description for table: market_1d


,ts,open,high,low,close,volume
count,44244,20388.000000,20388.000000,20388.000000,20388.000000,2.038800e+04
mean,2018-08-08 02:59:04.344995840,7248.647543,7289.738919,7204.866902,7250.029302,1.547526e+09
min,2010-01-01 00:00:00,0.000000,0.000000,0.000000,0.499000,0.000000e+00
25%,2014-10-29 18:00:00,13.940000,14.477500,13.380000,13.790000,0.000000e+00
50%,2018-11-13 00:00:00,2404.160034,2422.609985,2388.484985,2403.470093,2.651250e+08
75%,2022-07-17 06:00:00,11282.622803,11395.682617,11180.172363,11313.187500,3.155120e+09
max,2026-03-21 00:00:00,50243.148438,50512.789062,50115.031250,50188.140625,9.397454e+10
std,NaN,10686.781105,10744.624931,10625.732434,10688.666393,2.199283e+09



Description for table: market_15m


,ts,open,high,low,close,volume
count,18341,8546.000000,8546.000000,8546.000000,8546.000000,8.546000e+03
mean,2026-02-06 23:10:53.372226304,12957.589948,12971.270294,12944.336138,12957.634958,6.423450e+07
min,2025-12-28 23:00:00,3.958000,3.960000,3.956000,3.958000,0.000000e+00
25%,2026-01-16 18:00:00,15.872500,15.960000,15.820000,15.880000,0.000000e+00
50%,2026-02-06 08:30:00,27.375000,27.610001,27.090000,27.384999,0.000000e+00
75%,2026-02-26 16:15:00,23042.681152,23084.961914,23008.543945,23037.813477,8.369775e+07
max,2026-03-17 23:45:00,50500.710938,50512.789062,50379.578125,50501.019531,6.820269e+09
std,NaN,17888.054153,17906.446555,17870.343683,17888.141626,2.384175e+08


1-At least 50% of the data values are null.

2- There are huge value jumps, likely due to mixing different assets (symbols) in the same table.

3-The standard deviation is very high, indicating the data is not homogeneous.

> Next Steps / Notes:
At this stage, I encountered a lot of missing data. I need to compare the raw Parquet data with my database to ensure there is no data loss between the transform and load steps.

In [8]:
# Base directory containing subfolders
base_dir = r"C:\germeen\AlphaPulse\config\raw_data"

# Subfolders to check
subfolders = ["1d", "1h", "15m"]

# Loop through each subfolder
for folder in subfolders:
    folder_path = os.path.join(base_dir, folder)
    print("-" * 50)
    print(f"Checking {folder} data:")

    for file in os.listdir(folder_path):
        if file.endswith(".parquet"):
            path = os.path.join(folder_path, file)
            table = pq.ParquetFile(path)
            display(f"{file} → Rows: {table.metadata.num_rows}")

--------------------------------------------------
Checking 1d data:


'BTC-USD.parquet → Rows: 4204'

'caretDJI.parquet → Rows: 4078'

'caretGSPC.parquet → Rows: 4078'

'caretIXIC.parquet → Rows: 4078'

...

'caretIXIC.parquet → Rows: 1040'

'caretTNX.parquet → Rows: 1080'

'caretVIX.parquet → Rows: 2148'

'CLeqF.parquet → Rows: 3663'

'GCeqF.parquet → Rows: 3659'

In [9]:
for table in tables:
    print("=" * 50)
    print(f"Data quality check for table: {table}")
    
    query = f"""
    SELECT symbol, COUNT(*) AS row_count
    FROM {table}
    GROUP BY symbol;
    """
    
    df = pd.read_sql(query, engine)
    display(df.T)

Data quality check for table: market_1h


,0,1,2,3,4,5,6
symbol,GC=F,^GSPC,^VIX,^IXIC,^TNX,CL=F,^DJI
row_count,11810,3553,7333,3553,3612,11444,3588


Data quality check for table: market_1d


,0,1,2,3,4,5,6,7,8,9,10
symbol,GC=F,^GSPC,EURUSD=X,^VIX,^IXIC,^TNX,^DJI,CL=F,BTC-USD,GBPUSD=X,ETH-USD
row_count,4077,4078,4221,4078,4078,4076,4078,4078,4204,4221,3055


Data quality check for table: market_15m


,0,1,2,3,4,5,6
symbol,GC=F,^GSPC,^VIX,^IXIC,^TNX,^DJI,CL=F
row_count,4923,1404,2876,1404,1458,1404,4872


From this, we notice that the database contains more entries than the raw data. To understand where the extra data comes from, let’s check for duplicated values.

In [10]:
for table in tables:
    print("=" * 50)
    print(f"Checking duplicates in table: {table}")
    
    query = f"SELECT * FROM {table}"
    df = pd.read_sql(query, engine)
    
    duplicates = df.duplicated().sum()
    display(f"Number of duplicate rows: {duplicates}")

Checking duplicates in table: market_1h


'Number of duplicate rows: 0'

Checking duplicates in table: market_1d


'Number of duplicate rows: 0'

Checking duplicates in table: market_15m


'Number of duplicate rows: 0'

There are no duplicates, so the extra data in the database is due to new data being added alongside the old data.
- Maybe it is a timestamp-missing issue lets check

In [11]:
#check for duplicates in each file in the raw database tables
table_results = {}

for folder in subfolders:
    folder_path = os.path.join(base_dir, folder)
    results = []
    
    for file in os.listdir(folder_path):
        if file.endswith(".parquet"):
            path = os.path.join(folder_path, file)
            df = pd.read_parquet(path, engine="fastparquet")
            df["ts"] = pd.to_datetime(df["ts"])
            
            results.append({
                "file": file,
                "symbol": df["symbol"].iloc[0],
                "min_ts": df["ts"].min(),
                "max_ts": df["ts"].max(),
                "count": len(df)
            })
    
    # Save as a separate DataFrame for each table
    table_results[folder] = pd.DataFrame(results)

# Display each table separately
for folder, df in table_results.items():
    print("=" * 50)
    print(f"Summary for table/folder: {folder}")
    display(df)

Summary for table/folder: 1d


,file,symbol,min_ts,max_ts,count
0,BTC-USD.parquet,BTC-USD,2014-09-17 00:00:00+00:00,2026-03-21 00:00:00+00:00,4204
1,caretDJI.parquet,^DJI,2010-01-04 00:00:00+00:00,2026-03-20 00:00:00+00:00,4078
2,caretGSPC.parquet,^GSPC,2010-01-04 00:00:00+00:00,2026-03-20 00:00:00+00:00,4078
3,caretIXIC.parquet,^IXIC,2010-01-04 00:00:00+00:00,2026-03-20 00:00:00+00:00,4078
4,caretTNX.parquet,^TNX,2010-01-04 00:00:00+00:00,2026-03-20 00:00:00+00:00,4076
5,caretVIX.parquet,^VIX,2010-01-04 00:00:00+00:00,2026-03-20 00:00:00+00:00,4078
6,CLeqF.parquet,CL=F,2010-01-04 00:00:00+00:00,2026-03-20 00:00:00+00:00,4078
7,ETH-USD.parquet,ETH-USD,2017-11-09 00:00:00+00:00,2026-03-21 00:00:00+00:00,3055
8,EURUSDeqX.parquet,EURUSD=X,2010-01-01 00:00:00+00:00,2026-03-20 00:00:00+00:00,4221
9,GBPUSDeqX.parquet,GBPUSD=X,2010-01-01 00:00:00+00:00,2026-03-20 00:00:00+00:00,4221


Summary for table/folder: 1h


,file,symbol,min_ts,max_ts,count
0,caretDJI.parquet,^DJI,2024-03-18 13:30:00+00:00,2026-03-17 19:30:00+00:00,3483
1,caretGSPC.parquet,^GSPC,2024-03-18 13:30:00+00:00,2026-03-17 19:30:00+00:00,3483
2,caretIXIC.parquet,^IXIC,2024-03-18 13:30:00+00:00,2026-03-17 19:30:00+00:00,3483
3,caretTNX.parquet,^TNX,2024-03-18 12:20:00+00:00,2026-03-17 18:20:00+00:00,3507
4,caretVIX.parquet,^VIX,2024-03-18 07:00:00+00:00,2026-03-17 20:00:00+00:00,7123
5,CLeqF.parquet,CL=F,2024-03-18 04:00:00+00:00,2026-03-17 23:00:00+00:00,11214
6,GCeqF.parquet,GC=F,2024-03-18 04:00:00+00:00,2026-03-17 23:00:00+00:00,11465


Summary for table/folder: 15m


,file,symbol,min_ts,max_ts,count
0,caretDJI.parquet,^DJI,2026-01-20 14:30:00+00:00,2026-03-17 19:45:00+00:00,1040
1,caretGSPC.parquet,^GSPC,2026-01-20 14:30:00+00:00,2026-03-17 19:45:00+00:00,1040
2,caretIXIC.parquet,^IXIC,2026-01-20 14:30:00+00:00,2026-03-17 19:45:00+00:00,1040
3,caretTNX.parquet,^TNX,2026-01-20 13:20:00+00:00,2026-03-17 18:50:00+00:00,1080
4,caretVIX.parquet,^VIX,2026-01-19 08:15:00+00:00,2026-03-17 20:00:00+00:00,2148
5,CLeqF.parquet,CL=F,2026-01-19 04:45:00+00:00,2026-03-17 23:45:00+00:00,3663
6,GCeqF.parquet,GC=F,2026-01-19 04:30:00+00:00,2026-03-17 23:45:00+00:00,3659


In [12]:
#check for max and min timestamps in each file in the raw database tables
for table in tables:
    print("-" * 50)
    print(f"Checking table: {table} (min/max timestamps per symbol)")

    # Query to get min/max timestamps for each symbol
    query = f"""
    SELECT 
        symbol,
        COUNT(*) AS total_rows,
        MIN(ts) AS min_ts,
        MAX(ts) AS max_ts
    FROM {table}
    GROUP BY symbol
    ORDER BY total_rows DESC;
    """
    
    df = pd.read_sql(query, engine)
    display(df)

--------------------------------------------------
Checking table: market_1h (min/max timestamps per symbol)


,symbol,total_rows,min_ts,max_ts
0,GC=F,11810,2024-02-26 05:00:00,2026-03-17 23:00:00
1,CL=F,11444,2024-03-04 05:00:00,2026-03-17 23:00:00
2,^VIX,7333,2024-02-26 08:00:00,2026-03-17 20:00:00
3,^TNX,3612,2024-02-26 13:20:00,2026-03-17 18:20:00
4,^DJI,3588,2024-02-26 14:30:00,2026-03-17 19:30:00
5,^IXIC,3553,2024-03-04 14:30:00,2026-03-17 19:30:00
6,^GSPC,3553,2024-03-04 14:30:00,2026-03-17 19:30:00


--------------------------------------------------
Checking table: market_1d (min/max timestamps per symbol)


,symbol,total_rows,min_ts,max_ts
0,EURUSD=X,4221,2010-01-01,2026-03-20
1,GBPUSD=X,4221,2010-01-01,2026-03-20
2,BTC-USD,4204,2014-09-17,2026-03-21
3,^VIX,4078,2010-01-04,2026-03-20
4,^IXIC,4078,2010-01-04,2026-03-20
5,^DJI,4078,2010-01-04,2026-03-20
6,CL=F,4078,2010-01-04,2026-03-20
7,^GSPC,4078,2010-01-04,2026-03-20
8,GC=F,4077,2010-01-04,2026-03-20
9,^TNX,4076,2010-01-04,2026-03-20


--------------------------------------------------
Checking table: market_15m (min/max timestamps per symbol)


,symbol,total_rows,min_ts,max_ts
0,GC=F,4923,2025-12-28 23:00:00,2026-03-17 23:45:00
1,CL=F,4872,2025-12-28 23:00:00,2026-03-17 23:45:00
2,^VIX,2876,2025-12-29 08:15:00,2026-03-17 20:00:00
3,^TNX,1458,2025-12-29 13:20:00,2026-03-17 18:50:00
4,^IXIC,1404,2025-12-29 14:30:00,2026-03-17 19:45:00
5,^GSPC,1404,2025-12-29 14:30:00,2026-03-17 19:45:00
6,^DJI,1404,2025-12-29 14:30:00,2026-03-17 19:45:00


In [13]:
tables = ["market_1d", "market_1h", "market_15m"]
freq_map = {"market_1d": "1D", "market_1h": "1H", "market_15m": "15min"}

# Loop through each table
for table in tables:
    print("=" * 50)
    print(f"Checking missing timestamps in table: {table}")
    
    # Load the table
    query = f"SELECT * FROM {table}"
    df = pd.read_sql(query, engine)
    
    # Convert ts to datetime
    df["ts"] = pd.to_datetime(df["ts"])
    
    # Get the frequency for this table
    freq = freq_map[table]
    
    # Check missing timestamps per symbol
    symbols = df["symbol"].unique()
    missing_summary = []
    
    for sym in symbols:
        df_sym = df[df["symbol"] == sym].sort_values("ts")
        ts_full = pd.date_range(start=df_sym["ts"].min(), end=df_sym["ts"].max(), freq=freq)
        missing_count = len(ts_full.difference(df_sym["ts"]))
        missing_summary.append({"symbol": sym, "missing_count": missing_count})
    
    # Display results in a table
    missing_df = pd.DataFrame(missing_summary)
    display(missing_df)

Checking missing timestamps in table: market_1d


,symbol,missing_count
0,^VIX,1842
1,BTC-USD,0
2,CL=F,1842
3,ETH-USD,0
4,EURUSD=X,1702
5,GBPUSD=X,1702
6,GC=F,1843
7,^DJI,1842
8,^GSPC,1842
9,^IXIC,1842


Checking missing timestamps in table: market_1h


C:\Users\AdminOS\AppData\Local\Temp\ipykernel_23136\324010885.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  ts_full = pd.date_range(start=df_sym["ts"].min(), end=df_sym["ts"].max(), freq=freq)


,symbol,missing_count
0,GC=F,6233
1,^DJI,14418
2,CL=F,6431
3,^TNX,14394
4,^VIX,10680
5,^GSPC,14285
6,^IXIC,14285


Checking missing timestamps in table: market_15m


,symbol,missing_count
0,CL=F,2716
1,GC=F,2665
2,^DJI,6106
3,^GSPC,6106
4,^IXIC,6106
5,^TNX,6053
6,^VIX,4660


Large number of missing timestamps observed across symbols and intervals.
- Likely reasons:
1. Non-trading hours, weekends, or exchange holidays.
2. Less liquid symbols or data source limitations.
3. Partial historical data imports or ETL gaps.
- Note: These gaps reflect real trading schedules; handle with imputation 
or filtering if continuous intervals are needed.


In [14]:
#   OHLCV  validation

def validate_ohlc(tables, engine):
    """
    Validate OHLCV data for multiple tables in a database.

    Checks:
    - Missing values in OHLCV columns
    - High >= Low
    - Open and Close within Low and High
    - Volume >= 0
    - Duplicate rows
    """
    validation_results = []

    for table in tables:
        print("=" * 50)
        print(f"Validating table: {table}")

        # Load full table
        query = f"SELECT * FROM {table}"
        df = pd.read_sql(query, engine)

        symbols = df["symbol"].unique()

        for sym in symbols:
            df_sym = df[df["symbol"] == sym].copy()

            # Count missing values per column
            missing = df_sym[['open','high','low','close','volume']].isna().sum().to_dict()

            # Count High < Low
            high_low_errors = ((df_sym['high'] < df_sym['low'])).sum()

            # Count Open/Close outside High-Low
            open_errors = ((df_sym['open'] < df_sym['low']) | (df_sym['open'] > df_sym['high'])).sum()
            close_errors = ((df_sym['close'] < df_sym['low']) | (df_sym['close'] > df_sym['high'])).sum()

            # Count negative/zero volume
            vol_errors = (df_sym['volume'] <= 0).sum()

            # Count duplicate rows
            duplicate_rows = df_sym.duplicated().sum()

            validation_results.append({
                "table": table,
                "symbol": sym,
                "total_rows": len(df_sym),
                "missing_open": missing['open'],
                "missing_high": missing['high'],
                "missing_low": missing['low'],
                "missing_close": missing['close'],
                "missing_volume": missing['volume'],
                "high_lt_low": high_low_errors,
                "open_out_of_bounds": open_errors,
                "close_out_of_bounds": close_errors,
                "volume_invalid": vol_errors,
                "duplicate_rows": duplicate_rows
            })

    # Convert to DataFrame for a clean summary
    validation_df = pd.DataFrame(validation_results)
    return validation_df

# -------------------------------
# Example usage
# -------------------------------
result = validate_ohlc(tables, engine)
display(result)

Validating table: market_1d
Validating table: market_1h
Validating table: market_15m


,table,symbol,total_rows,missing_open,missing_high,missing_low,missing_close,missing_volume,high_lt_low,open_out_of_bounds,close_out_of_bounds,volume_invalid,duplicate_rows
0,market_1d,^VIX,4078,0,0,0,0,0,0,0,0,4078,0
1,market_1d,BTC-USD,4204,4204,4204,4204,4204,4204,0,0,0,0,0
2,market_1d,CL=F,4078,4078,4078,4078,4078,4078,0,0,0,0,0
3,market_1d,ETH-USD,3055,3055,3055,3055,3055,3055,0,0,0,0,0
4,market_1d,EURUSD=X,4221,4221,4221,4221,4221,4221,0,0,0,0,0
5,market_1d,GBPUSD=X,4221,4221,4221,4221,4221,4221,0,0,0,0,0
6,market_1d,GC=F,4077,4077,4077,4077,4077,4077,0,0,0,0,0
7,market_1d,^DJI,4078,0,0,0,0,0,0,0,0,0,0
8,market_1d,^GSPC,4078,0,0,0,0,0,0,0,0,1,0
9,market_1d,^IXIC,4078,0,0,0,0,0,0,0,0,0,0


Data Validation Summary:
- No duplicate rows were detected.
- A significant number of missing values exist due to missing timestamps; however, this poses no risk as there are no jumps in the data.
- OHLCV validation passed with no errors; all high, low, open, close, and volume values are consistent.
- The 'adj_close' column is unnecessary and should be removed.



# Feature engnireening

In [15]:
# Data Cleaning
# List of tables to clean
tables = ["market_1d", "market_1h", "market_15m"]


cleaned_tables = {}

for table in tables:
    print("="*50)
    print(f"Cleaning table: {table}")
    
    query = f"SELECT * FROM {table}"
    df = pd.read_sql(query, engine)
    
    # Convert timestamp
    df['ts'] = pd.to_datetime(df['ts'])
    
    # Drop adj_close if exists
    if 'adj_close' in df.columns:
        df = df.drop(columns=['adj_close'])
    
    # Sort by symbol and timestamp
    df = df.sort_values(['symbol', 'ts'])
    
   
    # Keep only complete OHLCV rows (no NaN)
    df = df.dropna(subset=['open', 'high', 'low', 'close', 'volume'])
    
    # Reset index
    df = df.reset_index(drop=True)
    
    cleaned_tables[table] = df
    print(f"Table {table} cleaned: {len(df)} rows remaining\n")

Cleaning table: market_1d
Table market_1d cleaned: 20388 rows remaining

Cleaning table: market_1h
Table market_1h cleaned: 21639 rows remaining

Cleaning table: market_15m
Table market_15m cleaned: 8546 rows remaining



Project Problem 

“Multi-Timeframe Financial Market Feature Engineering for Tabular Time Series Modeling”

Short Description:

The problem addresses building a structured dataset from multiple financial market data sources (daily, hourly, 15-minute) with inconsistent coverage across symbols and intervals. The goal is to create aggregated features per day from higher-resolution data, alongside daily features, to enable time-aware machine learning models for trend prediction or market analysis, without losing valuable data.

Key Points to Highlight in Documentation:

Data Sources: Yahoo Finance (1D, 1H, 15min) for multiple symbols.
Challenges:
Different timeframes have varying coverage (some symbols only exist in 1D).
Missing data in higher-resolution tables → cannot directly merge.
Need to preserve as much historical data as possible.
Solution Approach:
Aggregate 1H and 15min tables per day → compute micro/macro trends, returns, volatility.
Merge aggregated features with daily features → final tabular dataset.
Train ML models using tabular time series features instead of full sequence models for speed and practical portfolio-ready results.

In [19]:
# -------------------------------
# 1️⃣ Load your tables
# -------------------------------
df_1d = cleaned_tables["market_1d"]
df_1h = cleaned_tables["market_1h"]
df_15m = cleaned_tables["market_15m"]

# -------------------------------
# 2️⃣ 1D Feature Engineering
# -------------------------------
df_1d = df_1d.sort_values(['symbol', 'ts'])
df_1d['daily_return'] = (df_1d['close'] - df_1d['open']) / df_1d['open']

# Rename open/close/volume to match final column names
df_1d = df_1d.rename(columns={'open': 'daily_open', 'close': 'daily_close', 'volume': 'daily_volume'})

# Rolling features
df_1d['MA_7'] = df_1d.groupby('symbol')['daily_close'].transform(lambda x: x.rolling(7).mean())
df_1d['MA_30'] = df_1d.groupby('symbol')['daily_close'].transform(lambda x: x.rolling(30).mean())
df_1d['daily_volatility'] = df_1d.groupby('symbol')['daily_return'].transform(lambda x: x.rolling(10).std())
df_1d['trend'] = (df_1d['MA_7'] > df_1d['MA_30']).astype(int)

# -------------------------------
# 3️⃣ 1H Feature Engineering (aggregate per day)
# -------------------------------
df_1h = df_1h.sort_values(['symbol', 'ts'])
df_1h['date'] = df_1h['ts'].dt.date
df_1h['hourly_return'] = df_1h.groupby('symbol')['close'].pct_change()

agg_1h = df_1h.groupby(['symbol', 'date']).agg(
    hourly_return_mean=('hourly_return', 'mean'),
    hourly_return_std=('hourly_return', 'std'),
    last_hour_return=('hourly_return', lambda x: x.iloc[-1] if len(x) > 0 else np.nan)
).reset_index()

agg_1h['hourly_trend'] = (agg_1h['last_hour_return'] > 0).astype(int)

# -------------------------------
# 4️⃣ 15min Feature Engineering (aggregate per day)
# -------------------------------
df_15m = df_15m.sort_values(['symbol', 'ts'])
df_15m['date'] = df_15m['ts'].dt.date
df_15m['min15_return'] = df_15m.groupby('symbol')['close'].pct_change()

agg_15m = df_15m.groupby(['symbol', 'date']).agg(
    min15_return_mean=('min15_return', 'mean'),
    min15_return_std=('min15_return', 'std'),
    last_15min_return=('min15_return', lambda x: x.iloc[-1] if len(x) > 0 else np.nan)
).reset_index()

agg_15m['min15_trend'] = (agg_15m['last_15min_return'] > 0).astype(int)

# -------------------------------
# 5️⃣ Merge all features into final dataset
# -------------------------------
df_1d['date'] = df_1d['ts'].dt.date
final_df = df_1d.merge(agg_1h, on=['symbol', 'date'], how='left')
final_df = final_df.merge(agg_15m, on=['symbol', 'date'], how='left')

# Keep only necessary columns
final_columns = [
    'symbol', 'ts', 'daily_open', 'daily_close', 'daily_return',
    'MA_7', 'MA_30', 'trend', 'daily_volatility', 'daily_volume',
    'hourly_return_mean', 'hourly_return_std', 'last_hour_return', 'hourly_trend',
    'min15_return_mean', 'min15_return_std', 'last_15min_return', 'min15_trend'
]

final_df = final_df[final_columns]

# -------------------------------
# 6️⃣ Ready for ML
# -------------------------------
display(final_df.head())

,symbol,ts,daily_open,daily_close,daily_return,MA_7,MA_30,trend,daily_volatility,daily_volume,hourly_return_mean,hourly_return_std,last_hour_return,hourly_trend,min15_return_mean,min15_return_std,last_15min_return,min15_trend
0,^DJI,2010-01-04,10430.690430,10583.959961,0.014694,NaN,NaN,0,NaN,179780000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,^DJI,2010-01-05,10584.559570,10572.019531,-0.001185,NaN,NaN,0,NaN,188540000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,^DJI,2010-01-06,10564.719727,10573.679688,0.000848,NaN,NaN,0,NaN,186040000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,^DJI,2010-01-07,10571.110352,10606.860352,0.003382,NaN,NaN,0,NaN,217390000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,^DJI,2010-01-08,10606.400391,10618.190430,0.001112,NaN,NaN,0,NaN,172710000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
print(final_df.isnull().sum())

symbol                    0
ts                        0
daily_open                0
daily_close               0
daily_return              0
MA_7                     30
MA_30                   145
trend                     0
daily_volatility         46
daily_volume              0
hourly_return_mean    17818
hourly_return_std     17818
last_hour_return      17818
hourly_trend          17818
min15_return_mean     20118
min15_return_std      20118
last_15min_return     20118
min15_trend           20118
dtype: int64


In [21]:
display(final_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20388 entries, 0 to 20387
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   symbol              20388 non-null  object        
 1   ts                  20388 non-null  datetime64[ns]
 2   daily_open          20388 non-null  float64       
 3   daily_close         20388 non-null  float64       
 4   daily_return        20388 non-null  float64       
 5   MA_7                20358 non-null  float64       
 6   MA_30               20243 non-null  float64       
 7   trend               20388 non-null  int64         
 8   daily_volatility    20342 non-null  float64       
 9   daily_volume        20388 non-null  float64       
 10  hourly_return_mean  2570 non-null   float64       
 11  hourly_return_std   2570 non-null   float64       
 12  last_hour_return    2570 non-null   float64       
 13  hourly_trend        2570 non-null   float64   

None

In [27]:
# =========================================
# Feature Engineering Functions (independent)
# =========================================

def feature_engineering_1d(df):
    df = df.copy()
    df = df.sort_values(['symbol', 'ts'])
    
    df['daily_return'] = df.groupby('symbol')['close'].pct_change()
    df['MA_7'] = df.groupby('symbol')['close'].transform(lambda x: x.rolling(7).mean())
    df['MA_30'] = df.groupby('symbol')['close'].transform(lambda x: x.rolling(30).mean())
    df['daily_volatility'] = df.groupby('symbol')['daily_return'].transform(lambda x: x.rolling(7).std())
    df['trend'] = (df['MA_7'] > df['MA_30']).astype(int)
    df['regime'] = "sideways"
    df.loc[df['MA_7'] > df['MA_30'], 'regime'] = "bull"
    df.loc[df['MA_7'] < df['MA_30'], 'regime'] = "bear"
    
    return df

def feature_engineering_1h(df):
    df = df.copy()
    df = df.sort_values(['symbol', 'ts'])
    
    df['hourly_return'] = df.groupby('symbol')['close'].pct_change()
    df['hourly_mean'] = df.groupby('symbol')['close'].transform(lambda x: x.rolling(24).mean())
    df['hourly_volatility'] = df.groupby('symbol')['hourly_return'].transform(lambda x: x.rolling(24).std())
    df['hourly_trend'] = (df['hourly_return'] > 0).astype(int)
    
    return df

def feature_engineering_15m(df):
    df = df.copy()
    df = df.sort_values(['symbol', 'ts'])
    
    df['min15_return'] = df.groupby('symbol')['close'].pct_change()
    df['min15_mean'] = df.groupby('symbol')['close'].transform(lambda x: x.rolling(16).mean())
    df['min15_volatility'] = df.groupby('symbol')['min15_return'].transform(lambda x: x.rolling(16).std())
    df['min15_trend'] = (df['min15_return'] > 0).astype(int)
    
    return df

# =========================================
# Apply Feature Engineering
# =========================================

df_1d = feature_engineering_1d(cleaned_tables['market_1d'])
df_1h = feature_engineering_1h(cleaned_tables['market_1h'])
df_15m = feature_engineering_15m(cleaned_tables['market_15m'])

print("Feature Engineering applied:")
print(f"1D table: {df_1d.shape}")
print(f"1H table: {df_1h.shape}")
print(f"15min table: {df_15m.shape}")

Feature Engineering applied:
1D table: (20388, 14)
1H table: (21639, 12)
15min table: (8546, 12)


In [30]:
import streamlit as st
import pandas as pd
import plotly.express as px

# Load feature-engineered data (from previous step)
# df_1d, df_1h, df_15m

st.title("📊 Multi-Timeframe Market Dashboard")

# --- Select symbol ---
symbol_list = df_1d['symbol'].unique()
selected_symbol = st.selectbox("Select Symbol", symbol_list)

# Filter daily data
daily_df = df_1d[df_1d['symbol']==selected_symbol]

# Last day summary
last_day = daily_df.iloc[-1]
st.subheader(f"Latest Data for {selected_symbol}")
st.write(f"Close: {last_day['close']:.2f}")
st.write(f"Daily Return: {last_day['daily_return']:.4f}")
st.write(f"Trend: {'Bull' if last_day['trend']==1 else 'Bear'}")
st.write(f"Regime: {last_day['regime']}")

# --- Daily trends plot ---
fig_ma = px.line(daily_df, x='ts', y=['MA_7','MA_30'], title="MA_7 vs MA_30")
st.plotly_chart(fig_ma)

fig_vol = px.line(daily_df, x='ts', y='daily_volatility', title="Daily Volatility")
st.plotly_chart(fig_vol)

# Optional: show high-frequency aggregates if needed
if st.checkbox("Show 1H Aggregates"):
    df_hour = df_1h[df_1h['symbol']==selected_symbol]
    fig_hr = px.line(df_hour, x='ts', y='hourly_return_mean', title="Hourly Mean Returns")
    st.plotly_chart(fig_hr)

if st.checkbox("Show 15min Aggregates"):
    df_min15 = df_15m[df_15m['symbol']==selected_symbol]
    fig_15 = px.line(df_min15, x='ts', y='min15_return_mean', title="15min Mean Returns")
    st.plotly_chart(fig_15)

2026-03-22 00:38:37.518 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 00:38:37.987 
  command:

    streamlit run c:\germeen\AlphaPulse\jupyter_env\lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-22 00:38:37.990 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 00:38:37.991 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 00:38:37.991 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 00:38:37.991 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 00:38:37.993 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 00:38:37.993 Thread 'MainTh

In [29]:
pip install streamlit plotly

  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB 